# 01 - Quickstart

**Audience:** you just installed `eyelean_analysis` and want to know it works.

Run all cells. You should see a session summary, a scanpath plot, and a pupil
timeline. If any cell errors out, your environment isn't set up correctly
(usually a missing `pip install -e .` or a stale checkout).

## What you'll get

- Confirmation that the package imports and finds your data.
- One-line summary of the recording (n samples, duration, sample rate).
- Scanpath plot.
- Pupil timeline.
- Pointers to the deeper notebooks for each subsystem.

## Getting your data into this notebook

`notebook_context()` auto-discovers a CSV in this order:

1. **Explicit path** — `notebook_context(csv="path/to/EyeTracking_*.csv")`
2. **Environment variable** — `export EYELEAN_CSV=path/to/file.csv`
3. **`Logs/` folder** — most-recent main `EyeTracking_*.csv` in any
   `Logs/` directory walking up from this notebook.
4. **Bundled sample** — falls back to a v1.2 SampleExperiment recording
   shipped in `Assets/StreamingAssets/` so the notebook runs end-to-end
   on a fresh checkout before you've recorded anything yourself.

## 0. Bootstrap

In [ ]:
import os
import sys
from pathlib import Path

# Allow opening this notebook from a checkout without `pip install -e`.
_here = Path(os.getcwd()).resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "eyelean_analysis" / "__init__.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import eyelean_analysis as ela

ctx = ela.notebook_context()
print(ctx)

## 1. Recording at a glance

In [ ]:
data = ctx.data
print(f"Samples:     {len(data.df)}")
print(f"Duration:    {data.duration:.1f} s")
print(f"Sample rate: {data.get_sample_rate():.1f} Hz")
print(f"Phases:      {data.get_phases()}")

## 2. Scanpath

Combined gaze direction projected to (yaw, pitch) and plotted as a raw trajectory.

In [ ]:
df = data.df
if all(c in df.columns for c in ("combined_dir_x", "combined_dir_y", "combined_dir_z")):
    dx = df["combined_dir_x"].to_numpy()
    dy = df["combined_dir_y"].to_numpy()
    dz = df["combined_dir_z"].to_numpy()
    yaw = np.degrees(np.arctan2(dx, dz))
    pitch = -np.degrees(np.arcsin(np.clip(dy, -1, 1)))
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(yaw, pitch, linewidth=0.4, color="steelblue")
    ax.set_xlabel("yaw (deg)")
    ax.set_ylabel("pitch (deg)")
    ax.invert_yaxis()
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_title(f"Scanpath - {ctx.csv_path.name}")
    plt.tight_layout()
    plt.show()
else:
    print("No combined gaze direction columns; skipping scanpath.")

## 3. Pupil timeline

In [ ]:
if "left_pupil_diameter" in df.columns or "right_pupil_diameter" in df.columns:
    t = data.get_timestamps()
    fig, ax = plt.subplots(figsize=(12, 3))
    if "left_pupil_diameter" in df.columns:
        ax.plot(t, df["left_pupil_diameter"], label="left", linewidth=0.7)
    if "right_pupil_diameter" in df.columns:
        ax.plot(t, df["right_pupil_diameter"], label="right", linewidth=0.7, alpha=0.7)
    ax.set_xlabel("time (s)")
    ax.set_ylabel("pupil diameter (mm)")
    ax.set_title("Pupil timeline")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No pupil columns; skipping.")

## Where to next?

- **02_explore_session** - end-to-end walkthrough of one recording.
- **03_data_quality** - validity, gaps, blink rate.
- **04_eye_movements** - I-VT classifier, fixation/saccade stats, K-coefficient.
- **05_pupil_lhipa** - cognitive-load metric on the pupil signal.
- **06_sample_experiment** - per-phase report against the bundled SampleExperiment.
- **07_scene_sidecars** - v1.2 scene-state + events sidecars.
- **08_posthoc_correction** - re-apply a calibration profile to a CSV.
- **09_batch_processing** - many recordings to one summary.